In [53]:
!pip show scikit-learn
!pip show imbalanced-learn

Name: scikit-learn
Version: 1.5.2
Summary: A set of python modules for machine learning and data mining
Home-page: https://scikit-learn.org
Author: 
Author-email: 
License: BSD 3-Clause License

Copyright (c) 2007-2024 The scikit-learn developers.
All rights reserved.

Redistribution and use in source and binary forms, with or without
modification, are permitted provided that the following conditions are met:

* Redistributions of source code must retain the above copyright notice, this
  list of conditions and the following disclaimer.

* Redistributions in binary form must reproduce the above copyright notice,
  this list of conditions and the following disclaimer in the documentation
  and/or other materials provided with the distribution.

* Neither the name of the copyright holder nor the names of its
  contributors may be used to endorse or promote products derived from
  this software without specific prior written permission.

THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS 

In [55]:
!pip install imbalanced-learn==0.13.0

   ---------------------------------------- 0.0/238.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/238.4 kB ? eta -:--:--
   - -------------------------------------- 10.2/238.4 kB ? eta -:--:--
   ----- --------------------------------- 30.7/238.4 kB 325.1 kB/s eta 0:00:01
   ----- --------------------------------- 30.7/238.4 kB 325.1 kB/s eta 0:00:01
   ----- --------------------------------- 30.7/238.4 kB 325.1 kB/s eta 0:00:01
   ----- --------------------------------- 30.7/238.4 kB 325.1 kB/s eta 0:00:01
   ----- --------------------------------- 30.7/238.4 kB 325.1 kB/s eta 0:00:01
   ----- --------------------------------- 30.7/238.4 kB 325.1 kB/s eta 0:00:01
   ------ --------------------------------- 41.0/238.4 kB 98.1 kB/s eta 0:00:03
   ------ --------------------------------- 41.0/238.4 kB 98.1 kB/s eta 0:00:03
   ------------- ------------------------- 81.9/238.4 kB 169.9 kB/s eta 0:00:01
   --------------- ----------------------- 92.2/238.4 kB 174.7 kB/

In [19]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import re
import string
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder 
from imblearn.under_sampling import NearMiss
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

In [21]:
df = pd.read_csv(r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\fix_clean_data.csv")
df.head()

,id,text,durasi,emotion
0,1,tradisional laos kayak gimana kayak gini jadi ...,1.58,6
1,2,episode berikutnya akan dimulai sekarang kita ...,2.07,6
2,3,itu agak aneh ya biasanya pohon pepaya itu kan...,0.45,6
3,4,bulan ini sampah organik di rumah kita olah pa...,2.06,6
4,5,mobil yang akan kita pakai buat root party nan...,1.04,4


In [23]:
# Load data
def load_data(file_path):
    """Load CSV data"""
    df = pd.read_csv(r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\fix_clean_data.csv")
    return df

In [25]:
def preprocess_text_for_resampling(df, text_column='text'):
    """
    Preprocess text data untuk resampling
    Menggunakan TF-IDF untuk mengkonversi text menjadi numerical features
    """
    # Clean data terlebih dahulu
    print(f"Original data shape: {df.shape}")
    
    # Check for missing values
    print(f"Missing values in text column: {df[text_column].isnull().sum()}")
    print(f"Missing values in emotion column: {df['emotion'].isnull().sum()}")
    
    # Drop rows with missing values
    df_clean = df.dropna(subset=[text_column, 'emotion']).copy()
    print(f"Data shape after cleaning: {df_clean.shape}")
    
    # Convert text to string and remove empty strings
    df_clean[text_column] = df_clean[text_column].astype(str)
    df_clean = df_clean[df_clean[text_column].str.strip() != '']
    df_clean = df_clean[df_clean[text_column] != 'nan']
    print(f"Data shape after removing empty texts: {df_clean.shape}")
    
    # Check emotion distribution
    print("\nEmotion distribution:")
    print(df_clean['emotion'].value_counts())
    print()
    
    if len(df_clean) == 0:
        raise ValueError("No valid data remaining after cleaning!")
    
    # Initialize TF-IDF Vectorizer
    vectorizer = TfidfVectorizer(
        max_features=1000,  # Batasi fitur untuk efisiensi
        stop_words='english',
        ngram_range=(1, 2),
        min_df=1,  # Minimal document frequency
        max_df=0.95  # Maksimal document frequency
    )
    
    try:
        # Transform text to numerical features
        X = vectorizer.fit_transform(df_clean[text_column])
        y = df_clean['emotion']
        
        print(f"TF-IDF shape: {X.shape}")
        print(f"Feature names sample: {vectorizer.get_feature_names_out()[:10]}")
        
        return X, y, vectorizer, df_clean
        
    except Exception as e:
        print(f"Error in TF-IDF transformation: {str(e)}")
        print("Sample texts:")
        for i, text in enumerate(df_clean[text_column].head()):
            print(f"{i}: {repr(text)}")
        raise

In [27]:
def apply_nearmiss_and_smoothing(X, y, target_counts):
    """
    Apply NearMiss untuk Surprise dan SMOTE untuk emotions lainnya
    """
    # Konversi ke dense array untuk SMOTE
    X_dense = X.toarray()
    
    # Cek distribusi awal
    print("Distribusi emotion sebelum resampling:")
    print(Counter(y))
    print()
    
    # Pisahkan data berdasarkan emotion
    emotion_data = {}
    for emotion in y.unique():
        mask = y == emotion
        emotion_data[emotion] = {
            'X': X_dense[mask],
            'y': y[mask]
        }
    
    # Hasil akhir
    X_resampled = []
    y_resampled = []
    
    for emotion, target_count in target_counts.items():
        if emotion in emotion_data:
            X_emotion = emotion_data[emotion]['X']
            y_emotion = emotion_data[emotion]['y']
            current_count = len(y_emotion)
            
            print(f"Processing {emotion}: {current_count} -> {target_count}")
            
            if current_count > target_count:
                # Undersampling dengan NearMiss
                if current_count > 1:  # Pastikan ada cukup data untuk undersampling
                    # Buat data dummy untuk class lain (diperlukan untuk NearMiss)
                    other_emotions = [e for e in emotion_data.keys() if e != emotion]
                    if other_emotions:
                        # Ambil sampel kecil dari emotion lain
                        other_emotion = other_emotions[0]
                        X_other_small = emotion_data[other_emotion]['X'][:min(50, len(emotion_data[other_emotion]['X']))]
                        y_other_small = [other_emotion] * len(X_other_small)
                        
                        # Gabungkan untuk NearMiss
                        X_temp = np.vstack([X_emotion, X_other_small])
                        y_temp = np.hstack([y_emotion, y_other_small])
                        
                        # Apply NearMiss
                        nm = NearMiss(version=1, n_neighbors=min(3, current_count-1))
                        X_nm, y_nm = nm.fit_resample(X_temp, y_temp)
                        
                        # Ambil hanya data untuk emotion yang diproses
                        mask_emotion = y_nm == emotion
                        X_result = X_nm[mask_emotion]
                        y_result = y_nm[mask_emotion]
                        
                        # Jika masih terlalu banyak, random sample
                        if len(y_result) > target_count:
                            indices = np.random.choice(len(y_result), target_count, replace=False)
                            X_result = X_result[indices]
                            y_result = y_result[indices]
                    else:
                        # Jika tidak ada emotion lain, lakukan random undersampling
                        indices = np.random.choice(current_count, target_count, replace=False)
                        X_result = X_emotion[indices]
                        y_result = y_emotion.iloc[indices]
                else:
                    X_result = X_emotion
                    y_result = y_emotion
                    
            elif current_count < target_count:
                # Oversampling dengan SMOTE
                # Buat data dummy untuk SMOTE (minimal 2 classes diperlukan)
                other_emotions = [e for e in emotion_data.keys() if e != emotion]
                if other_emotions and current_count > 1:
                    other_emotion = other_emotions[0]
                    X_other_small = emotion_data[other_emotion]['X'][:min(current_count, len(emotion_data[other_emotion]['X']))]
                    y_other_small = [other_emotion] * len(X_other_small)
                    
                    # Gabungkan untuk SMOTE
                    X_temp = np.vstack([X_emotion, X_other_small])
                    y_temp = np.hstack([y_emotion, y_other_small])
                    
                    # Apply SMOTE

                    smote = SMOTE(
                        sampling_strategy={emotion: target_count},
                        k_neighbors=min(current_count-1, 5) if current_count > 1 else 1,
                        random_state=42
                    )
                    X_smote, y_smote = smote.fit_resample(X_temp, y_temp)
                    
                    # Ambil hanya data untuk emotion yang diproses
                    mask_emotion = y_smote == emotion
                    X_result = X_smote[mask_emotion]
                    y_result = y_smote[mask_emotion]
                else:
                    # Jika tidak bisa SMOTE, lakukan random oversampling
                    indices = np.random.choice(current_count, target_count, replace=True)
                    X_result = X_emotion[indices]
                    y_result = np.array([emotion] * target_count)
            else:
                # Jumlah sudah sesuai
                X_result = X_emotion
                y_result = y_emotion.values if hasattr(y_emotion, 'values') else np.array(y_emotion)
            
            X_resampled.append(X_result)
            y_resampled.append(y_result)
            
            print(f"Result for {emotion}: {len(y_result)} samples")
            print()
    
    # Gabungkan semua hasil
    X_final = np.vstack(X_resampled)
    y_final = np.hstack(y_resampled)
    
    return X_final, y_final


In [29]:
def reconstruct_dataframe(X_resampled, y_resampled, vectorizer, original_df):
    """
    Rekonstruksi DataFrame dengan data yang telah di-resample
    Note: Karena kita menggunakan TF-IDF, kita tidak bisa merekonstruksi teks asli
    Jadi kita akan membuat mapping berdasarkan similarity
    """
    # Untuk demo, kita akan membuat DataFrame sederhana
    # Dalam praktik nyata, Anda mungkin perlu menyimpan mapping text asli
    
    resampled_df = pd.DataFrame({
        'emotion': y_resampled,
        'resampled_features': [f"feature_vector_{i}" for i in range(len(y_resampled))]
    })
    
    return resampled_df

def main(csv_file_path):
    """Main function untuk menjalankan seluruh proses"""
    
    # Target counts sesuai requirement
    target_counts = {
        'Surprise': 160,
        'Joy': 130,
        'Anger': 130,
        'Sadness': 130,
        'Fear': 130,
        'Neutral': 130  # Asumsi ada emotion Neutral
    }
    
    # Load data
    print("Loading data...")
    df = load_data(csv_file_path)
    print(f"Data loaded: {len(df)} rows")
    print(f"Columns: {df.columns.tolist()}")
    print()
    
    # Preprocess
    print("Preprocessing text data...")
    try:
        X, y, vectorizer, df_clean = preprocess_text_for_resampling(df)
        print(f"Text converted to {X.shape[1]} features")
        print()
        
        # Update target counts berdasarkan emotion yang benar-benar ada
        available_emotions = set(y.unique())
        print("Available emotions in data:")
        for emotion in available_emotions:
            current_count = sum(y == emotion)
            print(f"  - {emotion}: {current_count} samples")
        
        # Mapping emotion numbers ke target counts
        # Anda perlu menyesuaikan mapping ini berdasarkan label emotion Anda
        # Contoh mapping (sesuaikan dengan data Anda):
        emotion_mapping = {
            0: 130,  # Sesuaikan dengan emotion yang diinginkan
            1: 130,
            2: 160,  # Jika ini adalah 'Surprise'
            3: 130,
            4: 130,
            5: 130,
            6: 130,  # Emotion dengan count tertinggi (296)
            7: 130   # Emotion dengan count tinggi (165)
        }
        
        # Buat updated_target_counts berdasarkan emotion yang ada
        updated_target_counts = {}
        for emotion in available_emotions:
            if emotion in emotion_mapping:
                updated_target_counts[emotion] = emotion_mapping[emotion]
            else:
                updated_target_counts[emotion] = 130  # Default
        
        print("\nTarget counts being used:")
        for emotion, count in updated_target_counts.items():
            current_count = sum(y == emotion)
            action = "undersample" if current_count > count else "oversample" if current_count < count else "no change"
            print(f"  Emotion {emotion}: {current_count} -> {count} ({action})")
        print()
        
    except Exception as e:
        print(f"Error in preprocessing: {str(e)}")
        return None, None, None
    
    # Apply resampling
    print("Applying NearMiss and SMOTE...")
    try:
        X_resampled, y_resampled = apply_nearmiss_and_smoothing(X, y, updated_target_counts)
        
        print("Distribusi emotion setelah resampling:")
        print(Counter(y_resampled))
        print()
        
        # Reconstruct DataFrame
        result_df = reconstruct_dataframe(X_resampled, y_resampled, vectorizer, df_clean)
        
        # Save results
        output_file = 'emotion_data_resampled.csv'
        result_df.to_csv(output_file, index=False)
        print(f"Results saved to: {output_file}")
        
        return result_df, X_resampled, y_resampled
        
    except Exception as e:
        print(f"Error in resampling: {str(e)}")
        import traceback
        traceback.print_exc()
        return None, None, None

# Contoh penggunaan
if __name__ == "__main__":
    # Ganti dengan path file CSV Anda
    csv_file_path = r"C:\Users\Aditya P J\Documents\Competition\Satria Data 2025\data\fix_clean_data.csv"
    
    try:
        result_df, X_resampled, y_resampled = main(csv_file_path)
        print("Resampling completed successfully!")
        
        if result_df is not None:
            # Tampilkan distribusi final
            print("\nFinal distribution:")
            print(result_df['emotion'].value_counts())
        else:
            print("No results to display.")
        
    except Exception as e:
        print(f"Error: {str(e)}")
        print("\nPastikan:")
        print("1. File CSV ada dan dapat diakses")
        print("2. File memiliki kolom 'text' dan 'emotion'")
        print("3. Install required packages: pip install pandas scikit-learn imbalanced-learn")

Loading data...
Data loaded: 722 rows
Columns: ['id', 'text', 'durasi', 'emotion']

Preprocessing text data...
Original data shape: (722, 4)
Missing values in text column: 0
Missing values in emotion column: 0
Data shape after cleaning: (722, 4)
Data shape after removing empty texts: (722, 4)

Emotion distribution:
emotion
6    296
7    165
4    137
2     51
0     35
5     17
1     15
3      6
Name: count, dtype: int64

TF-IDF shape: (722, 1000)
Feature names sample: ['acara' 'ada' 'ada di' 'ada juga' 'ada yang' 'adalah' 'aduh' 'agak' 'ai'
 'air']
Text converted to 1000 features

Available emotions in data:
  - 0: 35 samples
  - 1: 15 samples
  - 2: 51 samples
  - 3: 6 samples
  - 4: 137 samples
  - 5: 17 samples
  - 6: 296 samples
  - 7: 165 samples

Target counts being used:
  Emotion 0: 35 -> 130 (oversample)
  Emotion 1: 15 -> 130 (oversample)
  Emotion 2: 51 -> 160 (oversample)
  Emotion 3: 6 -> 130 (oversample)
  Emotion 4: 137 -> 130 (undersample)
  Emotion 5: 17 -> 130 (oversam